Here we are building a chatbot using Hugginface transformer pretrained model.

In [3]:
# Importing Libraries

import os
# Disable ALL progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()


In [4]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [5]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [6]:
!pip install transformers torch --quiet

In [7]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

Model loaded successfully.
Device map: Single device


Response function

In [8]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply
     

In [9]:

def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    # Initialize chat history with a system message to set behavior
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue
        # Generate reply from the model
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        # fallback
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?


User:  answer my questions


Chatbot: Of course! Please ask your questions and I'll do my best to answer them to the best of my ability.


User:  would you like a idea of making a chatbot using pretrained transformer model from hugging face


Chatbot: Certainly! Using a pretrained Transformer model from Hugging Face for creating a chatbot involves several steps. Below is a high-level overview of how you can achieve this:

### Prerequisites
1. **Install Required Libraries**
   ```bash
   pip install transformers torch
   ```

2. **Download Pretrained Model**
   You can download a pre-trained model from Hugging Face's Hub. For example, the `distilbert-base-uncased` model is quite popular for many NLP tasks.
   ```bash
   python -m transformers import_from_hub "distilbert-base-uncased"
   ```

3. **Prepare Your Data**
   Prepare your training data in a format that can be used by the model. Typically, this includes:
   - A list of sentences.
   - The corresponding labels (e.g., positive or negative sentiment).

4. **Create the Chatbot Class**
   Define a class to handle interactions with the user.

5. **Build the Model**
   Load the pretrained model and create a tokenizer.

6. **Train the Model (Optional)**
   If you want to fi

...

Chatbot: Glad you found the summary helpful! If you have any more questions or need further details, feel free to ask.


User:  no thats enough for now , see you next time , exit for now .


Chatbot: Goodbye! Take care.


User:  exit


Chatbot: Goodbye! Have a great day.
